In [ ]:
!pip install jax==0.6.0
!pip install flax
!pip install orbax-checkpoint==0.11.23
!pip install grain

In [ ]:
import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P
from flax import nnx
import flax
import orbax.checkpoint as ocp
import pandas as pd
import numpy as np
import optax
import math
import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
import grain
import pickle
from typing import Sequence

In [ ]:
jax.devices()

In [ ]:
MESH = jax.sharding.Mesh(
    devices=np.array(jax.devices()).reshape(2,1),
    axis_names=('x', "y"),
)
MESH

In [ ]:
IMAGE_SIZE = 32
PIXEL_LEVELS = 256
N_FILTERS = 128 * 2
RESIDUAL_BLOCKS = 7
BATCH_SIZE = 128
EPOCHS = 200
NUM_CHANNELS = 3
SAVE_DIR = 'output'

In [ ]:
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict
base_dir = "/kaggle/input/cifar10-python/cifar-10-batches-py"
batches = []
for i in range(5):
    batches.append(unpickle(base_dir + f"/data_batch_{i+1}"))
batches.append(unpickle(base_dir + f"/test_batch"))
print(len(batches))

In [ ]:
X, y = np.concatenate(
    list(map(lambda x: x[b"data"], batches)), 
    axis=0
), np.concatenate(
    list(map(lambda x: x[b"labels"], batches)),
    axis = 0
)
print(X.shape, y.shape)

In [ ]:
X.dtype

In [ ]:
class CifarDataSource(grain.sources.RandomAccessDataSource):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels
    
    def __getitem__(self, idx):
        return (self.images[idx, ...], self.labels[idx, ...])
        
    def __len__(self):
        return self.images.shape[0]

In [ ]:
TOTAL = X.shape[0]
TRAIN_SIZE = int(TOTAL * (0.8))
print(f"TRAIN_SIZE - {TRAIN_SIZE} \nVAL_SIZE - {TOTAL - TRAIN_SIZE}")

train_source = CifarDataSource(
    X[0:TRAIN_SIZE, ...],
    y[0:TRAIN_SIZE, ...]
)
val_source = CifarDataSource(
    X[TRAIN_SIZE:, ...],
    y[TRAIN_SIZE:, ...]
)

def preprocess_image(images: Sequence[np.ndarray]) -> tuple[np.ndarray]:
    imgs_int = np.reshape(images[0], (NUM_CHANNELS, IMAGE_SIZE, IMAGE_SIZE))
    imgs_int = np.moveaxis(imgs_int, 0, -1)
    
    imgs = (imgs_int / 256).astype(np.float32)
    
    imgs_int = (imgs_int / (256 / PIXEL_LEVELS)).astype(np.int32)
    # print(imgs.shape, imgs_int.shape)
    return imgs, imgs_int

def shard_images(x):
    return jax.device_put(
        x[0],
        jax.sharding.NamedSharding(
            MESH,
            P("x", None, None, None)
        )
    ), jax.device_put(
        x[1],
        jax.sharding.NamedSharding(
            MESH,
            P("x", None, None, None)
        )
    )
    
train_dataset = (
    grain.MapDataset.source(train_source)
    .shuffle(seed=60000)
    .map(preprocess_image)
    .batch(batch_size = BATCH_SIZE)
    .map(shard_images)
)
val_dataset = (
    grain.MapDataset.source(val_source)
    .shuffle(seed=60000)
    .map(preprocess_image)
    .batch(batch_size = BATCH_SIZE)
    .map(shard_images)
)

In [ ]:
print(len(train_dataset), len(val_dataset))

for index, (img, label) in enumerate(train_dataset):
    print(img.shape, img.sharding, label.shape, label.sharding)
    print(np.unique(label[100, ...]))
    fig, axs = plt.subplots(nrows=1, ncols = 2)
    axs[0].imshow(img[100, ...])
    plt.show()
    if index == 0:
        break

In [ ]:
from collections.abc import Iterable
from typing import Literal, Union, Sequence

class MaskedConv2D(nnx.Module):
    def __init__(
        self,
        mask_type: Literal["A", "B"],
        **kwargs
    ):
        super().__init__()

        assert isinstance(kwargs["kernel_size"], Iterable)
        assert len(kwargs["kernel_size"]) == 2

        self.mask_type = mask_type
        self.cnn_layer = nnx.Conv(**kwargs)
        self.prepare_mask()

    def prepare_mask(self,):
        kernel_shape = self.cnn_layer.kernel_size
        mask = np.zeros(kernel_shape)

        mask[:kernel_shape[0] // 2, ...] = 1.0
        mask[kernel_shape[0] // 2, :kernel_shape[1] // 2, ...] = 1.0

        if self.mask_type == "B":
            mask[kernel_shape[0] // 2, kernel_shape[1] // 2, ...] = 1.0

        mask = mask[..., np.newaxis, np.newaxis]
        mask = np.tile(
            mask, 
            (1, 1, self.cnn_layer.in_features, self.cnn_layer.out_features)
        )
        
        self.cnn_layer.mask = nnx.Variable(jnp.asarray(mask))
    
    def __call__(self, x):
        x = self.cnn_layer(x)
        return nnx.relu(x)

In [ ]:
class ResidualBlock(nnx.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        **kwargs
    ) -> None:
        super().__init__()

        self.conv_layer_1 = nnx.Conv(
            in_features = in_features,
            out_features = out_features,
            kernel_size = (1, 1),
            padding = "VALID",
            **kwargs
        )
        
        self.pixel_conv = MaskedConv2D(
            mask_type = "B",
            in_features = out_features,
            out_features = out_features // 2,
            kernel_size = (3, 3),
            padding="SAME",
            **kwargs
        )

        self.conv_layer_2 = nnx.Conv(
            in_features = out_features // 2,
            out_features = out_features,
            kernel_size = (1, 1),
            padding = "VALID",
            **kwargs
        )
    
    def __call__(self, inputs):
        x = self.conv_layer_1(inputs)
        x = nnx.relu(x)
        
        x = self.pixel_conv(x)
        x = nnx.relu(x)

        x = self.conv_layer_2(x)
        x = nnx.relu(x)

        return x + inputs

In [ ]:
class PixelCNN(nnx.Module):
    def __init__(
        self,
        in_features: int,
        num_residual_blocks: int,
        num_filters: int,
        pixel_levels: int,
        rngs: nnx.Rngs
    ):
        init_fn = nnx.initializers.lecun_normal()
        
        self.layer_1 = MaskedConv2D(
            mask_type = "A",
            in_features = in_features,
            out_features = num_filters,
            kernel_size = (7, 7),
            padding = "SAME" ,
            rngs = rngs,
            kernel_init = nnx.with_partitioning(
                init_fn,
                (None, None, None, "x")
            ),
        )

        self.resids_2 = []
        for _ in range(num_residual_blocks):
            self.resids_2.append(
                ResidualBlock(
                    in_features = num_filters,
                    out_features = num_filters,
                    rngs = rngs,
                    kernel_init = nnx.with_partitioning(
                        init_fn,
                        (None, None, None, "x")
                    ), 
                )
            )
        
        self.masked_3 = []
        for _ in range(2):
            self.masked_3.append(
                MaskedConv2D(
                    mask_type = "B",
                    in_features = num_filters,
                    out_features = num_filters,
                    kernel_size = (1, 1),
                    padding = "SAME",
                    rngs = rngs,
                    kernel_init = nnx.with_partitioning(
                        init_fn,
                        (None, None, None, "x")
                    ),
                )
            )
        
        self.final = nnx.Conv(
            in_features = num_filters,
            out_features = pixel_levels,
            kernel_size = (1, 1),
            padding = "VALID",
            rngs = rngs,
            kernel_init = nnx.with_partitioning(
                init_fn,
                (None, None, None, "x")
            ),
        )

    def __call__(self, x):
        x = self.layer_1(x)
        for layer_2 in self.resids_2:
            x = layer_2(x)
        
        x = nnx.relu(x)
        
        for layer_3 in self.masked_3:
            x = layer_3(x)
        
        return self.final(x)


In [ ]:
@nnx.jit
def create_sharded_model():
    model = PixelCNN(
        in_features = NUM_CHANNELS,
        num_residual_blocks = RESIDUAL_BLOCKS,
        num_filters = N_FILTERS,
        pixel_levels = PIXEL_LEVELS * NUM_CHANNELS,
        rngs = nnx.Rngs(42)
    )
    state = nnx.state(model)                   # The model's state, a pure pytree.
    pspecs = nnx.get_partition_spec(state)     # Strip out the annotations from state.
    sharded_state = jax.lax.with_sharding_constraint(state, pspecs)
    nnx.update(model, sharded_state)# The model is sharded now!
    return model

In [ ]:
with MESH:
  sharded_model = create_sharded_model()

nnx.display(sharded_model)

In [ ]:
class ImageGenerator(nnx.Module):
    def __init__(
        self,
        model: PixelCNN,
        input_shape: Sequence[int],
        num_images: int,
        pixel_levels: int
    ):
        self.model = model
        self.num_images = num_images
        self.pixel_levels = pixel_levels
        self.input_shape = input_shape

    def generate(self, temperature: float) -> np.ndarray:
        generated_images = np.zeros(shape=(self.num_images,) + self.input_shape)
        _, rows, cols, channels = generated_images.shape
        for row in range(rows):
            for col in range(cols):
                for channel in range(channels):
                    #Predict pixels one by one using the previously predicted pixels
                    probs = np.array(
                        jax.nn.softmax(
                            self.model(generated_images)[:, row, col, :]
                        )
                    ).reshape((-1, self.input_shape[-1], self.pixel_levels))[:, channel, :]
                    generated_images[:, row, col, channel] = [
                        self.sample_from(x, temperature) for x in probs
                    ]
                    generated_images[:, row, col, channel] /= self.pixel_levels
        return generated_images

    def sample_from(self, probs: jax.Array, temperature: float) -> np.ndarray:
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        
        probs[-1] = 1 - np.sum(probs[0:-1])
        return np.random.choice(len(probs), p=probs)

In [ ]:
from datetime import datetime

class ManageCheckpoints:
    def __init__(
        self,
        base_dir: str,
    ):
        self.ckpt_dir = ocp.test_utils.erase_and_create_empty(base_dir)
        self.checkpointer = ocp.StandardCheckpointer()

    def save_model(
        self,
        model: nnx.Module
    ) -> None:
        _, state = nnx.split(model)
        
        new_dir = self.ckpt_dir / f"{str(int(datetime.utcnow().timestamp()))}_state"
        self.checkpointer.save(new_dir, state)
        
        return str(new_dir)
        
    def load_model(
        self,
        model: nnx.Module,
        checkpoint: str,
        mesh: jax.sharding.Mesh = None
    ) -> nnx.Module:
        abstract_model = nnx.eval_shape(lambda: model)
        graphdef, abstract_state = nnx.split(abstract_model)
        nnx.display(abstract_state)

        if mesh:
            abstract_state = jax.tree.map(
                lambda a, s: jax.ShapeDtypeStruct(a.shape, a.dtype, sharding=s),
                abstract_state,
                nnx.get_named_sharding(abstract_state, mesh)
            )
        
        restored_state = self.checkpointer.restore(self.ckpt_dir / checkpoint, abstract_state)
        nnx.display(restored_state)
        
        return nnx.merge(graphdef, restored_state)

In [ ]:
class EarlyStopping:
    def __init__(
        self,
        monitor='val_loss',
        min_delta=0,
        patience=0,
        mode='auto',
        baseline=None,
        restore_best_weights=False,
        start_from_epoch=0
    ):
        pass
        

In [ ]:
from typing import Callable, Any
from collections import defaultdict
from IPython.display import clear_output
import matplotlib.pyplot as plt    

@nnx.jit
def train_step(
    model: nnx.Module,
    optimizer: nnx.Optimizer,
    metrics: nnx.MultiMetric,
    batch: Sequence[np.ndarray]
):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch)
    
    metrics.update(loss=loss, logits=logits, labels=batch[1])
    optimizer.update(grads, value=loss)


@nnx.jit
def eval_step(
    model: nnx.Module,
    metrics: nnx.MultiMetric,
    batch: Sequence[np.ndarray]
):
    loss, logits = loss_fn(model, batch)
    metrics.update(loss=loss, logits=logits, labels=batch[1])
    
def train(
    epochs: int,
    model: nnx.Module,
    optimizer: nnx.Optimizer,
    metrics: nnx.MultiMetric,
    train_dataset: grain.MapDataset,
    val_dataset: grain.MapDataset,
    train_metrics: dict,
    val_metrics: dict,
    checkpoint_manager: ManageCheckpoints = None,
    early_stopping: bool = False
):

    for epoch in range(1, epochs+1):
        
        model.train()
        epoch_train_metrics = defaultdict(list)
        with tqdm.tqdm(train_dataset, unit = "batch") as train_epochs:
            for index, batch in enumerate(train_epochs):
                train_epochs.set_description(f"Epoch {epoch} | Training | ")
                train_step(
                    model = model,
                    optimizer = optimizer,
                    metrics = metrics,
                    batch = batch
                )

                batch_metrics = metrics.compute()
                for metric, value in batch_metrics.items():
                    epoch_train_metrics[metric].append(value)
                metrics.reset()

                train_epochs.set_postfix(
                    **batch_metrics,
                    lr = optax.tree.get(optimizer.opt_state, "scale").value
                )
        for metric, value in epoch_train_metrics.items():
            train_metrics[metric].append(np.mean(value))

        model.eval()
        epoch_val_metrics = defaultdict(list)
        with tqdm.tqdm(val_dataset, unit = "batch") as val_epochs:
            for index, batch in enumerate(val_epochs):
                val_epochs.set_description(f"Epoch {epoch} | Validation | ")
                eval_step(
                    model = model,
                    metrics = metrics,
                    batch = batch
                )

                batch_metrics = metrics.compute()
                for metric, value in batch_metrics.items():
                    epoch_val_metrics[metric].append(value)
                metrics.reset()
                
                val_epochs.set_postfix(**batch_metrics)
        for metric, value in epoch_val_metrics.items():
            val_metrics[metric].append(np.mean(value))

        for metric in train_metrics.keys():
            print(f"{metric} --> \n Train: {train_metrics[metric]} \n Val: {val_metrics[metric]}")

        if checkpoint_manager:
            checkpoint_dir = checkpoint_manager.save_model(model)
            print(f"Saved Checkpoint at {checkpoint_dir}")
            
    return train_metrics, val_metrics


In [ ]:
def loss_fn(model: PixelCNN, batch):
    logits = model(batch[0])
    reshaped_logits = logits.reshape(batch[1].shape + (logits.shape[-1] // batch[1].shape[-1],))
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=reshaped_logits, labels=batch[1]
    ).mean()
    return loss, logits

LEARNING_RATE = 0.01
MOMENTUM = 0.99

PATIENCE = 3
PATIENCE_STEPS = math.ceil(TRAIN_SIZE / BATCH_SIZE) * PATIENCE
# PATIENCE_STEPS = 
print(f"Patience - {PATIENCE}, Patience Steps - {PATIENCE_STEPS}")

FACTOR=0.1
MIN_SCALE = 1e-7 / LEARNING_RATE

opt = optax.chain(
    optax.adam(LEARNING_RATE, MOMENTUM),
    optax.contrib.reduce_on_plateau(
        factor = FACTOR,
        patience = PATIENCE_STEPS,
        min_scale = MIN_SCALE
    )
)
# opt = optax.adam(LEARNING_RATE, MOMENTUM)

optimizer = nnx.Optimizer(
  sharded_model, opt, wrt=nnx.Param
)
metrics = nnx.MultiMetric(
  # accuracy=nnx.metrics.Accuracy(),
  loss=nnx.metrics.Average('loss'),
)

checkpoint_manager = ManageCheckpoints(
    base_dir = "/kaggle/working/checkpoints"
)

nnx.display(optimizer)

In [ ]:
train_metrics, val_metrics = defaultdict(list), defaultdict(list)

In [ ]:
train_metrics, val_metrics = train(
    epochs = EPOCHS,
    model = sharded_model,
    optimizer = optimizer,
    metrics = metrics,
    train_dataset = train_dataset,
    val_dataset = val_dataset,
    train_metrics = train_metrics,
    val_metrics = val_metrics,
    checkpoint_manager = checkpoint_manager,
    early_stopping = True
)

In [ ]:
def make_graphs(train_metrics, val_metrics):
    fig, axes = plt.subplots(1, len(train_metrics), figsize=(15, 5))
    for i, metric in enumerate(train_metrics.keys()):
        if len(train_metrics) == 1:
            axes.set_title(metric)
            axes.plot(train_metrics[metric], label=f'train_{metric}')
            axes.plot(val_metrics[metric], label=f'val_{metric}')
            axes.legend()
        else:
            axes[i].set_title(metric)
            axes[i].plot(train_metrics[metric], label=f'train_{metric}')
            axes[i].plot(val_metrics[metric], label=f'val_{metric}')
            axes[i].legend()
    plt.show()

In [ ]:
make_graphs(train_metrics, val_metrics)

In [ ]:
# image_generator = ImageGenerator(
#     model = sharded_model,
#     input_shape = (32, 32, 3),
#     num_images = 1,
#     pixel_levels = PIXEL_LEVELS
# )

In [ ]:
# arr = image_generator.generate(temperature = 1.0)
# plt.imshow(arr[0, ...])